Zadanie 11 – Kontekstowe embeddingi BERT

Zbadaj jak BERT radzi sobie z polisemia (wieloznacznosciam):
Wybierz 5 wieloznacznych slow (np. “bank”, “bat”, “bear”, “spring”, “rock”)
Dla kazdego slowa przygotuj 2 zdania z ROZNYM znaczeniem
Wyekstrahuj embedding slowa z BERT
Oblicz cosine similarity miedzy tymi samymi slowami w roznych kontekstach

Oczekiwany wynik: tabela cosine similarity, interpretacja

In [2]:
%pip install torch transformers

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from numpy.linalg import norm

# 1. Inicjalizacja tokenizera i modelu BERT
print("Pobieranie modelu BERT...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# 2. Definicja słów polisemicznych i ich kontekstów (zdań)
polysemous_words = {
    "bank": [
        "I need to deposit money in the bank.",       # Instytucja finansowa
        "We sat by the river bank to rest."           # Brzeg rzeki
    ],
    "bat": [
        "The flying bat caught an insect in the dark.", # Nietoperz
        "He swung the baseball bat really hard."        # Kij bejsbolowy
    ],
    "bear": [
        "We saw a huge brown bear in the forest.",    # Niedźwiedź
        "I cannot bear this heavy pain anymore."      # Znieść / wytrzymać
    ],
    "spring": [
        "The beautiful flowers bloom in the spring.", # Wiosna
        "The metal spring in the mattress is broken." # Sprężyna
    ],
    "rock": [
        "He threw a heavy rock into the water.",      # Kamień / skała
        "They played loud rock music all night."      # Muzyka rockowa
    ]
}

results = []

# 3. Ekstrakcja embeddingów i obliczanie podobieństwa
for word, sentences in polysemous_words.items():
    embeddings = []
    
    for sent in sentences:
        # Tokenizacja zdania
        inputs = tokenizer(sent, return_tensors="pt", padding=True)
        
        # Wyciągnięcie embeddingów z modelu (bez liczenia gradientów)
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Znalezienie indeksu badanego słowa
        tokens = tokenizer.tokenize(sent)
        word_idx = tokens.index(word) + 1  # +1 ponieważ na początku jest token [CLS]
        
        # Ekstrakcja wektora (embeddingu) dla konkretnego słowa z ostatniej ukrytej warstwy
        word_embedding = outputs.last_hidden_state[0, word_idx, :].numpy()
        embeddings.append(word_embedding)
        
    # Obliczanie Cosine Similarity między dwoma wektorami tego samego słowa
    emb1, emb2 = embeddings[0], embeddings[1]
    cos_sim = np.dot(emb1, emb2) / (norm(emb1) * norm(emb2))
    
    # Zapisanie wyniku
    results.append({
        "Słowo": word,
        "Znaczenie 1": sentences[0],
        "Znaczenie 2": sentences[1],
        "Cosine Similarity": round(cos_sim, 4)
    })

# 4. Wyświetlenie wyników w formie tabeli
df_results = pd.DataFrame(results)
print("\n=== Wyniki podobieństwa wektorów (Cosine Similarity) ===")
print(df_results.to_string(index=False))

     |████████████████████████████████| 73.6 MB 8.4 MB/s eta 0:00:01     |██████████▌                     | 24.2 MB 1.7 MB/s eta 0:00:29
     |████████████████████████████████| 12.0 MB 14.8 MB/s eta 0:00:01
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
     |████████████████████████████████| 200 kB 8.7 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 3.7 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 5.7 MB/s eta 0:00:01
     |████████████████████████████████| 676 kB 4.7 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 6.2 MB/s eta 0:00:01
  Using cached pyyaml-6.0.3-cp39-cp39-macosx_11_0_arm64.whl (174 kB)
     |████████████████████████████████| 566 kB 12.0 MB/s eta 0:00:01
     |████████████████████████████████| 447 kB 106.4 MB/s eta 0:00:01
     |████████████████████████████████| 288 kB 6.6 MB/s eta 0:00:01
     |████████████████████████████████| 3.8 MB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 536 kB 5.1 MB/s

/Users/martusia/Desktop/kurs-datascience/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/martusia/Desktop/kurs-datascience/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pobieranie modelu BERT...

=== Wyniki podobieństwa wektorów (Cosine Similarity) ===
 Słowo                                  Znaczenie 1                                 Znaczenie 2  Cosine Similarity
  bank         I need to deposit money in the bank.           We sat by the river bank to rest.             0.5170
   bat The flying bat caught an insect in the dark.      He swung the baseball bat really hard.             0.5635
  bear      We saw a huge brown bear in the forest.      I cannot bear this heavy pain anymore.             0.3608
spring   The beautiful flowers bloom in the spring. The metal spring in the mattress is broken.             0.4182
  rock        He threw a heavy rock into the water.      They played loud rock music all night.             0.5456
